In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
import librosa

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

In [ ]:
BASE_PATH = "dataset"

IMAGE_DIR  = f"{BASE_PATH}/images"
VOICE_DIR  = f"{BASE_PATH}/voice"
MOTION_DIR = f"{BASE_PATH}/motion"
PHYSIO_DIR = f"{BASE_PATH}/physio"

METADATA_JSON = f"{BASE_PATH}/autism_dataset_metadata.json"
ADOS_PATH = f"{BASE_PATH}/Toddler Autism dataset July 2018.csv"

In [ ]:
ados_df = pd.read_csv(ADOS_PATH)

ados_features = ados_df[
    [f"A{i}" for i in range(1,11)] + ["Age_Mons", "Sex"]
].copy()

ados_features["Sex"] = ados_features["Sex"].map({"m":0, "f":1})
ados_features = ados_features.fillna(0)

ADOS_DIM = ados_features.shape[1]

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import librosa

LABEL_MAP = {
    "typical": 0,
    "mild_asd": 1,
    "moderate_asd": 1,
    "severe_asd": 1
}

class AutismDataset(Dataset):
    def __init__(self, metadata_path, root_dir, ados_df, transform=None):
        with open(metadata_path) as f:
            self.samples = json.load(f)

        self.root = root_dir
        self.ados = ados_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def load_audio(self, path):
        y, sr = librosa.load(path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        return torch.tensor(mfcc).float()

    def load_motion(self, path):
        with open(path) as f:
            data = json.load(f)

        seqs = [np.array(v).flatten() for v in data.values()]
        max_len = max(len(v) for v in seqs)

        padded = [
            np.pad(v, (0, max_len - len(v))) for v in seqs
        ]

        return torch.tensor(np.stack(padded)).float()

    def load_physio(self, path):
        df = pd.read_csv(path)
        return torch.tensor(df.values).float()

    def __getitem__(self, idx):
        s = self.samples[idx]

        img_path = os.path.join(self.root, s["image"])
        aud_path = os.path.join(self.root, s["voice"])
        mot_path = os.path.join(self.root, s["motion"])
        phy_path = os.path.join(self.root, s["physio"])

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        aud = self.load_audio(aud_path)
        mot = self.load_motion(mot_path)
        phy = self.load_physio(phy_path)

        ados = torch.tensor(self.ados.iloc[idx].values).float()
        label = LABEL_MAP[s["label"]]

        return img, aud, mot, phy, ados, label

In [ ]:
dataset = AutismDataset(
    metadata_path=METADATA_JSON,
    root_dir=BASE_PATH,
    ados_df=ados_features,
    transform=image_transform
)

print("Total samples:", len(dataset))

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_ds, test_ds = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

train_labels = [dataset[i][5] for i in train_ds.indices]

class_counts = np.bincount(train_labels)
weights = 1. / class_counts

sample_weights = [weights[label] for label in train_labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=4, sampler=sampler)
test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False)

In [ ]:
model = MultimodalAutismModel().to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)model = MultimodalAutismModel().to(DEVICE)



In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()

    correct = 0
    total = 0
    running_loss = 0.0

    for img, aud, mot, phy, ados, y in train_loader:

        img = img.to(DEVICE)
        aud = aud.to(DEVICE)
        mot = mot.to(DEVICE)
        phy = phy.to(DEVICE)
        ados = ados.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        out, _ = model(img, aud, mot, phy, ados)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    acc = correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss:.4f} | Acc: {acc*100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

y_true, y_pred = [], []

model.eval()

with torch.no_grad():
    for img, aud, mot, phy, ados, y in test_loader:

        img = img.to(DEVICE)
        aud = aud.to(DEVICE)
        mot = mot.to(DEVICE)
        phy = phy.to(DEVICE)
        ados = ados.to(DEVICE)

        out, _ = model(img, aud, mot, phy, ados)

        pred = out.argmax(1)

        y_true.extend(y.numpy())
        y_pred.extend(pred.cpu().numpy())

print(classification_report(y_true, y_pred))

In [ ]:
torch.save(model.state_dict(), "autism_model.pt")
print("Model saved successfully")